In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split,cross_val_score,GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier,RandomForestClassifier
from sklearn.metrics import accuracy_score,recall_score,precision_score,roc_auc_score,classification_report,confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib


In [ ]:
#loading data

df = pd.read_csv(r'C:\Users\indur\OneDrive\Desktop\neural\fraud_detection\feature_engineering.csv')
print(df.to_string())

In [ ]:
# FEATURE VALUES AND TARGET VALUES

x = df.drop('fraud',axis=1)
y = df['fraud']

In [ ]:
# NUM_COL AND CAT_COL

numeric_features = x.select_dtypes(include=['int64','float64']).columns
categorical_features = x.select_dtypes(include=['object']).columns    

In [ ]:
# preprocessing columns 

preprocess = ColumnTransformer([

    ('num_col',StandardScaler(),numeric_features),
    ('cat_col',OneHotEncoder(handle_unknown='ignore'),categorical_features)

])


In [ ]:
# train_test_split

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=43,stratify=y)

In [ ]:
# LOGISTIC PIPELINE

logistic_pipeline = Pipeline([

    ('preprocessing',preprocess),
    ('log_model',LogisticRegression(max_iter=100,class_weight='balanced'))

])

# RANDOM FOREST PIPE LINES


random_pipeline = Pipeline([

    ('preprocessing',preprocess),
    ('reg_model',RandomForestClassifier(max_depth=5,random_state=42,max_samples=10,n_estimators=200,class_weight='balanced'))

])


# GRADIENT BOOSTING PIPE LINES

gradient_pipeline = Pipeline([

    ('preprocessing',preprocess),
    ('grd_model',GradientBoostingClassifier())

])

In [ ]:
models = {

    'logistic_regression':logistic_pipeline,
    'random_forest':random_pipeline,
    'Gradient_boosting':gradient_pipeline

}


In [ ]:
for name,model in models.items():
    print(f'\n{name}')

    model.fit(x_train,y_train)
    y_preds = model.predict(x_test)
    y_pred_prob = model.predict_proba(x_test)[:,1]

    threshold = 0.30
    y_pred = (y_pred_prob > threshold).astype(int)


    acc_score = accuracy_score(y_test,y_pred)
    print('accuracy_score:',acc_score)

    re_score = recall_score(y_test,y_pred)
    print('recall_score:',re_score)

    con_metix = confusion_matrix(y_test,y_pred)
    print('confusion_matrix:',con_metix)

    tp,tn,fp,fn = con_metix.ravel()
    print('True positive:',tp)
    print('True negative:',tn)
    print("False positive",fp)
    print('False negative:',fn)
    
    prec_score = precision_score(y_test,y_pred)
    print('precision_score:',prec_score)

    class_score = classification_report(y_test,y_pred)
    print('classification_report:',class_score)

    r_a_score = roc_auc_score(y_test,y_pred_prob)
    print('roc_auc_score:',r_a_score)



In [ ]:
result = x_test.copy()
result['Actual'] = y_test.values
result['predict'] = y_pred

print(result.to_string())


In [ ]:
#cross validation score

for name,model in models.items():
    print(f'\n{name}')

    score = cross_val_score(model,x,y,cv=5)
    print(score)
    print(score.mean())

In [ ]:
# feature importance

print(f'\n{'-------------------------------------------------FEATURE IMPORTANCES---------------------------------------'}')

model = logistic_pipeline.named_steps['log_model']
imp = model.coef_[0]
print('LOGISTIC_REGRESSION:',imp)


model_ran = random_pipeline.named_steps['reg_model']
random_feature = model_ran.feature_importances_
print('RANDOM_FOREST:',random_feature)


model_gb = gradient_pipeline.named_steps['grd_model']
importance = model_gb.feature_importances_
print('GRADIENT_FOREST:',importance)


In [ ]:
#hyper paramater

param_grid = {
    'reg_model__n_estimators':[100,300],
    'reg_model__max_depth':[10,30,None],
    'reg_model__min_samples_split':[2,5]

}

grid_search = GridSearchCV(random_pipeline,param_grid,cv=5,scoring='f1')
grid_search.fit(x_train,y_train)
print(grid_search.best_params_)

In [ ]:
param_grid = {

    'grd_model__n_estimators':[100,400],
    'grd_model__learning_rate':[0.1,0.1],
    'grd_model__max_depth':[10,30],
    'grd_model__min_samples_split':[2,4]
}

grid_search = GridSearchCV(gradient_pipeline,param_grid,cv=5,scoring='f1')
grid_search.fit(x_train,y_train)
print(grid_search.best_params_)

In [ ]:
joblib.dump(Pipeline,'Pipelines.pkl')
joblib.dump(models,'fraud_models.pkl')

In [ ]:
joblib.dump(random_pipeline,'fraud_model.pkl')
